In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir('/content')

!git clone https://github.com/VAL-Jerono/KHS_housing_dissertation.git
os.chdir('KHS_housing_dissertation')

import sys
sys.path.insert(0, 'src')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
fatal: destination path 'KHS_housing_dissertation' already exists and is not an empty directory.


In [ ]:
import pandas as pd

from pathlib import Path

RAW     = Path("/content/drive/MyDrive/KHS_Dissertation/data/raw")

PARQUET = Path("/content/drive/MyDrive/KHS_Dissertation/data/parquet")

PARQUET.mkdir(parents=True, exist_ok=True)

DTA_FILES = {

    "household":    "Household_Information_Data.dta",

    "individual":   "Individual_Data.dta",

    "dwelling":     "Dwelling_Units_Data.dta",

    "mortgage":     "Housing_Mortgage_Data.dta",

    "loan":         "Housing_Loans_Data.dta",

    "county":       "County_Physical_Planning_Data.dta",

    "real_estate":  "Real_Estate_Dataset.dta",

    "land":         "Land_Parcels_Data.dta",

    "institution":  "KHS_Institutional_Data.dta",

    "financiers":   "Housing_Financiers_Data.dta",

    "water":        "Water_Services_Providers_Data.dta",

    "nema":         "NEMA_Data_Set.dta",

    "housing_types":"Type of Housing Units.dta",

    "project":      "Project Information.dta",

}

for key, fname in DTA_FILES.items():

    src  = RAW / fname

    dest = PARQUET / fname.replace(".dta", ".parquet")

    if not src.exists():

        print(f"⚠  {key}: not found in raw/")

        continue

    if dest.exists():

        print(f"✓  {key}: already converted")

        continue

    print(f"→  {key}: converting...", end=" ")

    df = pd.read_stata(src, convert_categoricals=False)

    df.to_parquet(dest, index=False)

    print(f"{dest.stat().st_size/1e6:.1f} MB")

print("\nDone.")

✓  household: already converted
✓  individual: already converted
✓  dwelling: already converted
✓  mortgage: already converted
✓  loan: already converted
✓  county: already converted
✓  real_estate: already converted
✓  land: already converted
✓  institution: already converted
✓  financiers: already converted
✓  water: already converted
✓  nema: already converted
✓  housing_types: already converted
✓  project: already converted

Done.


In [ ]:
# Convert all int keys to strings recursively
def stringify_keys(obj):
    if isinstance(obj, dict):
        return {str(k): stringify_keys(v) for k, v in obj.items()}
    return obj

clean_val_labels = stringify_keys(val_labels)

with open(PARQUET / "household_value_labels.json", "w") as f:
    json.dump(clean_val_labels, f, indent=2)

print("Saved successfully.")

# Preview a few entries so you can see the structure
sample_keys = list(clean_val_labels.keys())[:5]
for k in sample_keys:
    print(f"\n{k}:")
    for code, label in clean_val_labels[k].items():
        print(f"  {code} → {label}")

Saved successfully.

A01:
  1 → Mombasa
  2 → Kwale
  3 → Kilifi
  4 → Tana River
  5 → Lamu
  6 → Taita-Taveta
  7 → Garissa
  8 → Wajir
  9 → Mandera
  10 → Marsabit
  11 → Isiolo
  12 → Meru
  13 → Tharaka-Nithi
  14 → Embu
  15 → Kitui
  16 → Machakos
  17 → Makueni
  18 → Nyandarua
  19 → Nyeri
  20 → Kirinyaga
  21 → Murang'a
  22 → Kiambu
  23 → Turkana
  24 → West Pokot
  25 → Samburu
  26 → Trans Nzoia
  27 → Uasin Gishu
  28 → Elgeyo-Marakwet
  29 → Nandi
  30 → Baringo
  31 → Laikipia
  32 → Nakuru
  33 → Narok
  34 → Kajiado
  35 → Kericho
  36 → Bomet
  37 → Kakamega
  38 → Vihiga
  39 → Bungoma
  40 → Busia
  41 → Siaya
  42 → Kisumu
  43 → Homabay
  44 → Migori
  45 → Kisii
  46 → Nyamira
  47 → Nairobi City
  96 → Outside Kenya

L30:
  0 → No
  1 → Yes

L29:
  0 → No
  1 → Yes

L28:
  -1 → Never Done Maintenance

L26:
  0 → No
  1 → Yes


In [ ]:
print("\n=== VARIABLE LABELS (column → question) ===\n")
for col, label in var_labels.items():
    print(f"{col:25s} → {label}")


=== VARIABLE LABELS (column → question) ===

interview__key            → Interview key (identifier in XX-XX-XX-XX format)
interview__id             → Unique 32-character long identifier of the interview
a01                       → County
countycode                → 
a07_1                     → Residence
serial                    → random_part
a12                       → Total persons in household
c01_1                     → C01_1.What is the main source of drinking water for members of your household?
c01_1other                → C01_1other:main source of drinking water for members of your household? Other
c01_2                     → C01_2 How do you access this drinking water?
c01_2other                → C01_2. How do you access this drinking water?
c01_3                     → C01_3Do you experience frequent water shortages?
c01_4                     → C01_4How long does is take to go there, get water and come back?
c01_5                     → Who usually goes to collect water from th

In [ ]:
import polars as pl

hh = pl.read_parquet(PARQUET / "Household_Information_Data.parquet")

print(f"Shape: {hh.shape}")
print(f"\nCounties: {hh['a01'].n_unique()}")
print(f"Urban/Rural split:\n{hh['a07_1'].value_counts()}")

# Null audit — most important thing to run
null_audit = (
    hh.null_count()
      .unpivot(variable_name="column", value_name="nulls")
      .with_columns((pl.col("nulls") / len(hh) * 100).round(1).alias("pct_null"))
      .sort("nulls", descending=True)
)

print("\nTop 40 columns by missingness:")
print(null_audit.head(40))

Shape: (21347, 392)

Counties: 47
Urban/Rural split:
shape: (2, 2)
┌───────┬───────┐
│ a07_1 ┆ count │
│ ---   ┆ ---   │
│ i8    ┆ u32   │
╞═══════╪═══════╡
│ 2     ┆ 9447  │
│ 1     ┆ 11900 │
└───────┴───────┘

Top 40 columns by missingness:
shape: (40, 3)
┌────────┬───────┬──────────┐
│ column ┆ nulls ┆ pct_null │
│ ---    ┆ ---   ┆ ---      │
│ str    ┆ u32   ┆ f64      │
╞════════╪═══════╪══════════╡
│ k10    ┆ 21333 ┆ 99.9     │
│ k32    ┆ 21285 ┆ 99.7     │
│ k24_1  ┆ 21212 ┆ 99.4     │
│ k27    ┆ 21212 ┆ 99.4     │
│ l24    ┆ 21198 ┆ 99.3     │
│ …      ┆ …     ┆ …        │
│ sf     ┆ 20785 ┆ 97.4     │
│ k14    ┆ 20738 ┆ 97.1     │
│ j21    ┆ 20729 ┆ 97.1     │
│ k08_2  ┆ 20637 ┆ 96.7     │
│ k08_3  ┆ 20637 ┆ 96.7     │
└────────┴───────┴──────────┘


In [ ]:
import json

with open(PARQUET / "household_variable_labels.json") as f:
    var_labels = json.load(f)

# Search for anything relevant to insurance, finance, expenditure
keywords = ['insur', 'premium', 'cover', 'policy', 'rent', 'income',
            'expend', 'spend', 'pay', 'loan', 'mortgage', 'afford']

print("Columns matching financial/insurance keywords:\n")
for col, label in var_labels.items():
    if any(k in label.lower() or k in col.lower() for k in keywords):
        print(f"  {col:20s} → {label}")

Columns matching financial/insurance keywords:

  c09__1               → Unable to pay monthly bill
  c09__2               → Unable to pay for electricity main grid connection
  c14_1                → C14_1:What is the average monthly expenditure
  d20__1               → High cost of rent
  d20__4               → Don't qualify for a loan
  d20__6               → High interest rates on loan/mortgage
  d20__8               → Low income
  h01                  → H01: accessibility of your current housing
  j09                  → J09: In your opinion, should government or public agencies regulate house rents?
  j11                  → J11: Should government or public agencies regulate interest rates on Loans
  j12_1                → J12_1: Should government or public agencies regulate interest rates on Loans
  j13                  → J13:Are you aware of the Affordable Housing Scheme/Programme?
  j17                  → awareness of the affordable housing relief which is 15%
  j18             

In [ ]:
# Configure git (first time only)
!git config user.email "gronjerono@gmail.com"
!git config user.name "VAL-Jerono"

# Stage notebook and outputs (not data)
!git add KHS.ipynb
!git status

fatal: pathspec 'KHS.ipynb' did not match any files
On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean
